In [92]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

In [93]:
#General examination of the dataset and dropping duplicates
df = pd.read_csv('movies_metadata.csv.gz')
#print(df.columns)
print(df.shape)
#print(df.isnull().sum())
df=df.drop_duplicates().reset_index(drop=True)

(45466, 24)


In [94]:
# selecting the columns that are useful to us
df = df[['title','overview','genres','tagline','vote_average','popularity']]
print(df.head())

                         title  \
0                    Toy Story   
1                      Jumanji   
2             Grumpier Old Men   
3            Waiting to Exhale   
4  Father of the Bride Part II   

                                            overview  \
0  Led by Woody, Andy's toys live happily in his ...   
1  When siblings Judy and Peter discover an encha...   
2  A family wedding reignites the ancient feud be...   
3  Cheated on, mistreated and stepped on, the wom...   
4  Just when George Banks has recovered from his ...   

                                              genres  \
0  [{'id': 16, 'name': 'Animation'}, {'id': 35, '...   
1  [{'id': 12, 'name': 'Adventure'}, {'id': 14, '...   
2  [{'id': 10749, 'name': 'Romance'}, {'id': 35, ...   
3  [{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...   
4                     [{'id': 35, 'name': 'Comedy'}]   

                                             tagline  vote_average popularity  
0                                        

In [95]:
#removing entries where there is no title
df = df.dropna(subset=['title'])

In [96]:
#replace na values with empty space in overview
df['overview'] = df['overview'].fillna('')

In [97]:
#replace na values with empty space in taglines
df['tagline'] = df['tagline'].fillna('')

In [98]:
#checking for na values
df.isnull().sum()

title           0
overview        0
genres          0
tagline         0
vote_average    0
popularity      0
dtype: int64

In [99]:
#examining the genre column
df.iloc[0]['genres']

"[{'id': 16, 'name': 'Animation'}, {'id': 35, 'name': 'Comedy'}, {'id': 10751, 'name': 'Family'}]"

In [100]:
#modifying genre column
import ast
df['genres']=df['genres'].apply(lambda x:" ".join([i['name'] for i in ast.literal_eval(x)]))

In [101]:
#combining all our data in one column
df['tags'] = df['overview'] + " " + df['genres'] + " " + df['tagline']

In [102]:
#text data in tags column before removing stopwords and lemmatizing tags
df['tags'][1]

"When siblings Judy and Peter discover an enchanted board game that opens the door to a magical world, they unwittingly invite Alan -- an adult who's been trapped inside the game for 26 years -- into their living room. Alan's only hope for freedom is to finish the game, which proves risky as all three find themselves running from giant rhinoceroses, evil monkeys and other terrifying creatures. Adventure Fantasy Family Roll the dice and unleash the excitement!"

In [103]:
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import re


In [104]:
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Aditya\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Aditya\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [105]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

In [106]:
#this function removes stopwords and lemmatizes the text
def preprocess_text(text):
  #lowercase
  text=str(text).lower()
  #remove puntuation
  text = re.sub(r'[^\w\s]', '', text)
  words = text.split()
  #remove stopwords
  words = [word for word in words if word not in stop_words]
  #lemmatize
  words = [lemmatizer.lemmatize(word) for word in words]
  return " ".join(words)

In [107]:
#Applying the function to tags
df['tags']=df['tags'].apply(preprocess_text)

In [108]:
# text data in tags after removing stopwords and lemmatizing tags
df['tags'][1]

'sibling judy peter discover enchanted board game open door magical world unwittingly invite alan adult who trapped inside game 26 year living room alans hope freedom finish game prof risky three find running giant rhinoceros evil monkey terrifying creature adventure fantasy family roll dice unleash excitement'

# Text Vectorization and Cosine Similarity

In [109]:
# Index Resetting and writing a new Index for each movie. Indices Used later for Cosine Similarity
df = df.reset_index(drop=True)
indices = pd.Series(df.index,index = df['title']).drop_duplicates()

### TF-IDF Vectorization

In [110]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(stop_words='english', max_features=50000, ngram_range=(1,2)) # Call Tfidf vectorizer
tfidf_mat = tfidf.fit_transform(df['tags']) #fit and then transform on tags, store in tfidf_mat

In [111]:
tfidf_mat #print tfidf_mat

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 1555624 stored elements and shape (45447, 50000)>

### Cosine Similarity

In [112]:
from sklearn.metrics.pairwise import cosine_similarity

# Create a function for recommendation
def recommendation(title, n=10):
    # If the title doesn't exist in the indices return "Movie not found"
    if title not in indices:
        return ['Movie Not Found!']
    #Get the index of the given title
    idx = indices[title] 
    # Get the vector of the given title stored at index idx and find similarity score to all other vectors in tfidf_mat
    sim_score = cosine_similarity(tfidf_mat[idx], tfidf_mat).flatten()
    # Sort the movies based on similarity score and get the top 10 recommendations
    sim_idx =  sim_score.argsort()[::-1][1:n+1]
    # Return the titles
    return df['title'].iloc[sim_idx] 

In [113]:
# Example Usage

recommendation('Toy Story')

# NOTE: THIS IS NOT A SEARCH ENGINE, THIS WILL ONLY GIVE RECOMMENDATIONS BASED ON THE MOVIES THAT ARE AVAILABLE IN THE DATABASE. 
# THE STRING SHOULD MATCH EXACTLY AS PER THE TITLE

2996                Toy Story 2
15344               Toy Story 3
24512                 Small Fry
28972           Superstar Goofy
17184                 Group Sex
6434     What's Up, Tiger Lily?
11396    For Your Consideration
1931                  Condorman
1071      Rebel Without a Cause
485                      Malice
Name: title, dtype: object

### Importing Pickle Files

In [114]:
import pickle

pickle.dump(tfidf_mat, open('tfidf_matrix.pkl', 'wb'))

pickle.dump(indices,open('indices.pkl', 'wb'))

df.to_pickle('df.pkl')

pickle.dump(tfidf, open('tfidf.pkl', 'wb'))